##### Copyright 2019 DeepMind Technologies Limited.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Reverb Tutorial (numpy, in-process)

This colab demonstrates Reverb's **embedded / numpy-only** mode: a
`reverb.Server(in_process=True)` holding tables directly in-process, with no
gRPC overhead and no TensorFlow dependency. Data flows as numpy arrays.

The same `Table` / `selectors` / `rate_limiters` / `TrajectoryWriter` APIs
also work over a networked `Server(in_process=False)` with the gRPC `Client`;
see the README for details.

# Setup

Install Reverb (built from this repository) and import it. Only numpy is
needed beyond Reverb itself.

In [ ]:
!pip install numpy dm-tree portpicker

In [ ]:
import numpy as np
import reverb

The code below defines a dummy RL environment for use in the examples. It is
pure numpy: observations are `[10, 10]` uint8 frames and actions are `[2]`
float32 vectors.

In [ ]:
OBSERVATION_SHAPE = (10, 10)
OBSERVATION_DTYPE = np.uint8
ACTION_SHAPE = (2,)
ACTION_DTYPE = np.float32

def agent_step(unused_timestep) -> np.ndarray:
  return (np.random.uniform(size=ACTION_SHAPE) > 0.5).astype(ACTION_DTYPE)

def environment_step(unused_action) -> np.ndarray:
  return np.random.randint(
      0, 256, size=OBSERVATION_SHAPE, dtype=OBSERVATION_DTYPE)

# Creating a Server and Client

Start an in-process server with a single prioritized table. `in_process=True`
means the server holds the tables directly and `server.in_process_client`
returns a `LocalClient` that talks to them with zero network overhead.

In [ ]:
simple_server = reverb.Server(
    tables=[
        reverb.Table(
            name='my_table',
            sampler=reverb.selectors.Prioritized(priority_exponent=0.8),
            remover=reverb.selectors.Fifo(),
            max_size=int(1e6),
            # Sets Rate Limiter to a low number for the examples.
            rate_limiter=reverb.rate_limiters.MinSize(2),
        )
    ],
    in_process=True,
)

client = simple_server.in_process_client
print(client.server_info())

# Example 1: Overlapping Trajectories

## Inserting Overlapping Trajectories

A `TrajectoryWriter` keeps a circular buffer of recent data references. An
item references a slice of that buffer, so successive items can overlap.

The writer is a context manager. `append` adds a step; `create_item` inserts
an item referencing slices of `writer.history`; `flush` blocks until the
server confirms the insertions.

In [ ]:
# Dynamically adds trajectories of length 3 to 'my_table' using a writer.
with client.trajectory_writer(num_keep_alive_refs=3) as writer:
  timestep = environment_step(None)
  for step in range(4):
    action = agent_step(timestep)
    writer.append({'action': action, 'observation': timestep})
    timestep = environment_step(action)

    if step >= 2:
      # The item consists of the 3 most recent timesteps, priority 1.5.
      writer.create_item(
          table='my_table',
          priority=1.5,
          trajectory={
              'actions': writer.history['action'][-3:],
              'observations': writer.history['observation'][-3:],
          }
      )

  writer.flush()

## Sampling Overlapping Trajectories

`client.sample` yields `ReplaySample` objects. With `emit_timesteps=False`
each sampled item is returned as a single `ReplaySample` whose `.data` is the
flat list of column arrays. Use `unpack_as_table_signature=True` (and a table
signature) to get the data back in the nested structure it was written with.

In [ ]:
for sample in client.sample('my_table', num_samples=2, emit_timesteps=False):
  print('info:', sample.info)
  # data is the flat list of columns: [actions_array, observations_array].
  print('actions shape:', np.asarray(sample.data[0]).shape)
  print('observations shape:', np.asarray(sample.data[1]).shape)

# Example 2: Complete Episodes

Create a fresh server for this example. Here each item is a whole episode.

The final timestep has an observation but no action. `append` tolerates a
partial step (only some columns present); the writer fills missing columns
with `None`, so the trajectory must exclude them.

In [ ]:
EPISODE_LENGTH = 150

episode_server = reverb.Server(
    tables=[
        reverb.Table(
            name='my_table',
            sampler=reverb.selectors.Prioritized(priority_exponent=0.8),
            remover=reverb.selectors.Fifo(),
            max_size=int(1e6),
            rate_limiter=reverb.rate_limiters.MinSize(2),
        )
    ],
    in_process=True,
)
client = episode_server.in_process_client

In [ ]:
NUM_EPISODES = 10

# Episodes are at most 150 steps, plus the terminal observation -> 151.
with client.trajectory_writer(
    num_keep_alive_refs=151) as writer:
  for _ in range(NUM_EPISODES):
    timestep = environment_step(None)

    for _ in range(EPISODE_LENGTH):
      action = agent_step(timestep)
      writer.append({'action': action, 'observation': timestep})
      timestep = environment_step(action)

    # Append the terminal observation WITHOUT an action. The writer fills
    # the action column with None, so the trajectory must drop it.
    writer.append({'observation': timestep})

    writer.create_item(
        table='my_table',
        priority=1.5,
        trajectory={
            'actions': writer.history['action'][:-1],
            'observations': writer.history['observation'][:],
        })

    # Blocks until the item is inserted; then clears the history buffer.
    writer.end_episode(timeout_ms=1000)

    assert len(writer.history['action']) == 0
    assert len(writer.history['observation']) == 0

In [ ]:
# Each sample is an entire episode.
for sample in client.sample('my_table', num_samples=2, emit_timesteps=False):
  print('actions shape:', np.asarray(sample.data[0]).shape)
  print('observations shape:', np.asarray(sample.data[1]).shape)

# Example 3: Multiple Priority Tables

A single server can hold multiple tables. Items referencing the same data
elements can live in different tables simultaneously.

In [ ]:
multitable_server = reverb.Server(
    tables=[
        reverb.Table(
            name='my_table_a',
            sampler=reverb.selectors.Prioritized(priority_exponent=0.8),
            remover=reverb.selectors.Fifo(),
            max_size=int(1e6),
            rate_limiter=reverb.rate_limiters.MinSize(2),
        ),
        reverb.Table(
            name='my_table_b',
            sampler=reverb.selectors.Uniform(),
            remover=reverb.selectors.Fifo(),
            max_size=int(1e6),
            rate_limiter=reverb.rate_limiters.MinSize(2),
        ),
    ],
    in_process=True,
)
client = multitable_server.in_process_client

## Inserting Sequences of Varying Length into Multiple Priority Tables

One writer can create items in several tables from the same buffered data.

In [ ]:
with client.trajectory_writer(num_keep_alive_refs=3) as writer:
  timestep = environment_step(None)
  for step in range(4):
    action = agent_step(timestep)
    writer.append({'action': action, 'observation': timestep})
    timestep = environment_step(action)

    if step >= 2:
      # Length-3 trajectory into the prioritized table.
      writer.create_item(
          table='my_table_a',
          priority=1.5,
          trajectory={
              'actions': writer.history['action'][-3:],
              'observations': writer.history['observation'][-3:],
          })
      # Length-2 trajectory into the uniform table, same underlying data.
      writer.create_item(
          table='my_table_b',
          priority=1.0,
          trajectory={
              'actions': writer.history['action'][-2:],
              'observations': writer.history['observation'][-2:],
          })

  writer.flush()

print('table_a items:', client.server_info()['my_table_a'].current_size)
print('table_b items:', client.server_info()['my_table_b'].current_size)

# Example 4: Samplers and Removers

Any selector can be used for sampling or removal. Combine them with
`max_times_sampled` and a rate limiter to build queues, stacks, heaps, and
circular buffers.

## A Queue and a Circular Buffer

`Table.queue` is shorthand for `Fifo` sampler + `Fifo` remover +
`max_times_sampled=1` + a `Queue` rate limiter: each item is sampled exactly
once then removed.

In [ ]:
queue_and_buffer_server = reverb.Server(
    tables=[
        # A FIFO queue: sample once, then remove.
        reverb.Table.queue(name='my_queue', max_size=1000),
        # A circular buffer: uniform sample, FIFO remove, resampleable.
        reverb.Table(
            name='my_buffer',
            sampler=reverb.selectors.Uniform(),
            remover=reverb.selectors.Fifo(),
            max_size=1000,
            rate_limiter=reverb.rate_limiters.MinSize(1),
        ),
    ],
    in_process=True,
)
client = queue_and_buffer_server.in_process_client
print(list(client.server_info().keys()))

# Example 5: Rate Limiters

Rate limiters block inserts and/or samples until conditions are met, which is
how Reverb paces producers and consumers.

`SampleToInsertRatio` keeps the long-run ratio of samples to inserts near a
target, blocking whichever side gets ahead. `timeout_ms` on `sample` turns a
block into a `reverb.errors.DeadlineExceededError` instead of waiting forever.

In [ ]:
rate_limited_server = reverb.Server(
    tables=[
        reverb.Table(
            name='my_table',
            sampler=reverb.selectors.Uniform(),
            remover=reverb.selectors.Fifo(),
            max_size=1000,
            rate_limiter=reverb.rate_limiters.SampleToInsertRatio(
                samples_per_insert=1.0, min_size_to_sample=2,
                error_buffer=1.0),
        )
    ],
    in_process=True,
)
client = rate_limited_server.in_process_client

# Sampling before min_size_to_sample is reached blocks, then times out.
try:
  next(client.sample('my_table', num_samples=1, timeout_ms=200,
                     emit_timesteps=False))
except reverb.errors.DeadlineExceededError:
  print('blocked as expected: not enough items yet')

# Example 6: Checkpointing

A `Client.checkpoint()` serializes the server's tables to disk. A new server
constructed with a `DefaultCheckpointer` pointing at the same root directory
loads the most recent checkpoint on startup.

> The on-disk format is length-delimited protobuf; checkpoints written by the
> old TensorFlow-based Reverb are **not** compatible and must be regenerated.

In [ ]:
import tempfile

ckpt_root = tempfile.mkdtemp(prefix='reverb_ckpt_')

def _ckpt_table():
  return reverb.Table(
      name='q', sampler=reverb.selectors.Fifo(),
      remover=reverb.selectors.Fifo(), max_size=10,
      max_times_sampled=1,
      rate_limiter=reverb.rate_limiters.MinSize(1))

# Write some data and checkpoint.
ckpt_server = reverb.Server(
    tables=[_ckpt_table()],
    in_process=True,
    checkpointer=reverb.platform.checkpointers_lib.DefaultCheckpointer(
        path=ckpt_root))
c = ckpt_server.in_process_client
with c.trajectory_writer(num_keep_alive_refs=1) as w:
  for i in range(5):
    w.append({'v': np.array([i], dtype=np.float32)})
    w.create_item(
        table='q', priority=1.0,
        trajectory={'v': w.history['v'][-1:]})
  w.flush()
path = c.checkpoint()
print('checkpoint written to:', path)
del ckpt_server

# Restore into a new server. DefaultCheckpointer takes the checkpoint root
# directory and loads the most recent checkpoint beneath it on startup.
restored_server = reverb.Server(
    tables=[_ckpt_table()],
    in_process=True,
    checkpointer=reverb.platform.checkpointers_lib.DefaultCheckpointer(
        path=ckpt_root))
rc = restored_server.in_process_client
print('restored table size:', rc.server_info()['q'].current_size)
for sample in rc.sample('q', num_samples=5, emit_timesteps=False):
  print('restored value:', np.asarray(sample.data[0]).reshape(-1)[0])

## Shutdown

Stop any remaining servers to release their background worker threads.

In [ ]:
simple_server.stop()
episode_server.stop()
multitable_server.stop()
queue_and_buffer_server.stop()
rate_limited_server.stop()
restored_server.stop()
